# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':   
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'click_stream.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'click_stream_bersih.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.scan_csv(DATA_PATH, try_parse_dates=True)

if df.collect_schema()['event_time'] == pl.String:
    df = df.with_columns(
        pl.col('event_time').str.to_datetime(
            format='%Y-%m-%dT%H:%M:%S%.fZ', time_unit='us', time_zone='UTC', strict=False
        )
    )

jumlah_baris = df.select(pl.len()).collect().item()
print(f'Ukuran awal: {jumlah_baris:,} baris x {len(df.collect_schema())} kolom')
print(df.collect_schema())
df.head().collect()

Ukuran awal: 12,833,602 baris x 6 kolom
Schema({'session_id': String, 'event_name': String, 'event_time': Datetime(time_unit='us', time_zone='UTC'), 'event_id': String, 'traffic_source': String, 'event_metadata': String})


session_id,event_name,event_time,event_id,traffic_source,event_metadata
str,str,"datetime[μs, UTC]",str,str,str
"""fb0abf9e-fd1a-44dd-b5c0-2834d5…","""HOMEPAGE""",2019-09-06 15:54:32.821085 UTC,"""9c4388c4-c95b-4678-b5ca-e9cbc0…","""MOBILE""",null
"""fb0abf9e-fd1a-44dd-b5c0-2834d5…","""SCROLL""",2019-09-06 16:03:57.821085 UTC,"""4690e1f5-3f99-42d3-84a5-22c4c4…","""MOBILE""",null
"""7d440441-e67a-4d36-b324-80ffd6…","""HOMEPAGE""",2019-09-01 12:05:10.322763 UTC,"""88aeaeb5-ec98-4859-852c-8abb48…","""MOBILE""",null
"""7d440441-e67a-4d36-b324-80ffd6…","""ADD_TO_CART""",2019-09-01 12:06:33.322763 UTC,"""934e306e-ecc6-472f-9ccb-12c853…","""MOBILE""","""{'product_id': 15315, 'quantit…"
"""7d440441-e67a-4d36-b324-80ffd6…","""BOOKING""",2019-09-01 12:15:29.425431 UTC,"""9f4767a1-40fa-4c9c-9524-dfad18…","""MOBILE""","""{'payment_status': 'Success'}"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [14]:
connection = duckdb.connect()
connection.execute("SET TimeZone = 'UTC'")

# View: DuckDB membaca file CSV langsung, tanpa menyalin data
connection.execute(f"""
    CREATE OR REPLACE VIEW dataset AS
    SELECT * FROM read_csv('{DATA_PATH.as_posix()}', header = true)
""")

profil = connection.sql('SUMMARIZE SELECT * FROM dataset').pl()
profil

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,null,str,str,str,i64,"decimal[9,2]"
"""session_id""","""VARCHAR""","""00001f29-3ede-4445-95f0-017893…","""fffffaf6-55ee-428b-a3bf-075cd7…",951924,null,null,null,null,null,12833602,0.00
"""event_name""","""VARCHAR""","""ADD_PROMO""","""SEARCH""",9,null,null,null,null,null,12833602,0.00
"""event_time""","""TIMESTAMP WITH TIME ZONE""","""2016-06-30 22:59:36.254358+00""","""2022-08-01 00:49:59.408424+00""",12842774,"""2020-10-17 06:11:58.513397+00""",null,"""2019-11-17 13:06:20.85093+00""","""2021-02-16 00:37:57.630062+00""","""2021-12-14 20:10:11.285776+00""",12833602,0.00
"""event_id""","""VARCHAR""","""0000001d-f3aa-498c-ad33-10e1fc…","""fffffc8c-b087-4025-8dfb-769e8b…",12716664,null,null,null,null,null,12833602,0.00
"""traffic_source""","""VARCHAR""","""MOBILE""","""WEB""",2,null,null,null,null,null,12833602,0.00
"""event_metadata""","""VARCHAR""","""{'payment_status': 'Failed'}""","""{'search_keywords': 'Topi'}""",2009445,null,null,null,null,null,12833602,66.58


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [15]:
null_summary = (
    df.select([pl.col(c).null_count().alias(c) for c in df.collect_schema().names()])
    .collect()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""event_metadata""",8544062,66.58


In [16]:
null_per_event = (
    df.group_by('event_name')
    .agg(
        pl.len().alias('jumlah_event'),
        (pl.col('event_metadata').is_null().mean() * 100).round(2).alias('persen_metadata_null'),
    )
    .sort('jumlah_event', descending=True)
    .collect()
)
null_per_event

event_name,jumlah_event,persen_metadata_null
str,u32,f64
"""CLICK""",2498038,100.0
"""HOMEPAGE""",2487126,100.0
"""ADD_TO_CART""",1937157,0.0
"""SCROLL""",1663571,100.0
"""ITEM_DETAIL""",1290396,100.0
"""SEARCH""",1173266,0.0
"""BOOKING""",852582,0.0
"""PROMO_PAGE""",604931,100.0
"""ADD_PROMO""",326535,0.0


**Interpretasi:**

- Terdapat lima event navigasi memiliki persentase null tepat **100%**, yaitu `CLICK` (2.498.038 baris), `HOMEPAGE` (2.487.126 baris), `SCROLL` (1.663.571 baris), `ITEM_DETAIL` (1.290.396 baris), dan `PROMO_PAGE` (604.931 baris).
- Dan ada empat event yang membawa detail aksi/transaksi memiliki persentase null tepat **0%**, yaitu `ADD_TO_CART` (1.937.157 baris), `SEARCH` (1.173.266 baris), `BOOKING` (852.582 baris), dan `ADD_PROMO` (326.535 baris).
- Kesimpulannya null pada `event_metadata` bersifat **struktural** karena pemisahannya tegas di angka 100% dan 0% sesuai jenis event—event navigasi memang tidak mencatat metadata tambahan, sedangkan seluruh event pencarian, keranjang, promo, dan pembayaran selalu terisi lengkap. Oleh karena itu, nilai null ini dipertahankan.

## 3. Penanganan Missing Values

Strategi bergantung pada jenis null. Null **struktural** pada `event_metadata` dibiarkan, karena mengisinya akan merusak makna data. Yang dibuang hanya baris dengan null pada **kolom kunci**, karena event tanpa identitas atau waktu tidak dapat dianalisis.

In [17]:
KOLOM_WAJIB = ['session_id', 'event_name', 'event_time', 'event_id', 'traffic_source']

df_missing_handled = df.filter(pl.all_horizontal(pl.col(KOLOM_WAJIB).is_not_null()))

jumlah_setelah_missing = df_missing_handled.select(pl.len()).collect().item()
print(f'Baris sebelum: {jumlah_baris:,}')
print(f'Baris sesudah: {jumlah_setelah_missing:,}')
print(f'Baris dibuang: {jumlah_baris - jumlah_setelah_missing:,}')

Baris sebelum: 12,833,602
Baris sesudah: 12,833,602
Baris dibuang: 0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [ ]:
sesi = (
    df_missing_handled
    .group_by('session_id')
    .agg(pl.len().alias('jumlah_event'))
    .collect()
)

KOLOM_NUMERIK = 'jumlah_event'

q1 = sesi[KOLOM_NUMERIK].quantile(0.25)
q3 = sesi[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr
p99 = sesi[KOLOM_NUMERIK].quantile(0.99)

n_outlier = sesi.filter(pl.col(KOLOM_NUMERIK) > batas_atas).height

print(f'Jumlah sesi: {sesi.height:,}')
print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}, p99: {p99}')
print(f'Sesi di atas batas atas IQR: {n_outlier:,} ({n_outlier / sesi.height * 100:.2f}%)')

sesi.sort(KOLOM_NUMERIK, descending=True).head(10)

Jumlah sesi: 895,203
Q1=6.0, Q3=18.0, IQR=12.0
Batas bawah: -12.0, Batas atas: 36.0, p99: 65.0
Sesi di atas batas atas IQR: 49,482 (5.53%)


session_id,jumlah_event
str,u32
"""65383bfb-5535-445f-ab52-aec465…",650
"""eb310630-8d65-4c59-8a33-a386b8…",650
"""12489f38-0fc1-4b80-9fe4-b267c4…",539
"""d46877e2-9e20-4277-8698-647dee…",539
"""c22cb71e-f232-4930-9d59-85340b…",519
"""230b1fd1-595a-4d28-a75f-23ff68…",483
"""756d0b19-5870-4bb8-aa74-611f85…",477
"""17347e17-4a5a-4233-9763-ced580…",461
"""5842f8ad-0de9-44fe-9033-5f847d…",460


In [19]:
# Tidak ada perubahan nilai pada tahap ini (outlier dilaporkan, bukan di-clip atau dihapus)
df_outlier_handled = df_missing_handled

## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [20]:
KOLOM_KUNCI = 'event_id'

total_baris = jumlah_setelah_missing
baris_unik = df_outlier_handled.select(pl.col(KOLOM_KUNCI).n_unique()).collect().item()
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

# Hapus duplikat hanya bila memang ada (menghemat memori pada data besar)
if total_baris - baris_unik > 0:
    df_dedup = df_outlier_handled.unique(
        subset=[KOLOM_KUNCI],
        keep='first',
        maintain_order=True,
    )
else:
    df_dedup = df_outlier_handled

Total baris: 12,833,602
Baris unik berdasarkan event_id: 12,833,602
Duplikat: 0


## 6. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

`sink_csv` menjalankan seluruh rencana lazy dan menulis hasilnya ke disk secara streaming, sehingga seluruh data tidak perlu dimuat ke RAM.

In [21]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.sink_csv(OUTPUT_PATH)

print(f'Data bersih tersimpan di: {OUTPUT_PATH}')
print(f'Ukuran file mentah : {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')
print(f'Ukuran file bersih : {OUTPUT_PATH.stat().st_size / 1_000_000:.1f} MB')

Data bersih tersimpan di: /home/jovyan/work/data/processed/click_stream_bersih.csv
Ukuran file mentah : 1715.3 MB
Ukuran file bersih : 1766.6 MB


## 7. Validasi Data Dasar

Aturan validasi dijalankan pada **file hasil simpan** (bukan data di memori), sehingga yang diuji adalah output sebenarnya. Setiap aturan menghitung jumlah **pelanggaran**; setelah cleaning seharusnya 0.

In [22]:
connection.execute(f"""
    CREATE OR REPLACE VIEW dataset_bersih AS
    SELECT * FROM read_csv('{OUTPUT_PATH.as_posix()}', header = true)
""")

validasi = connection.sql("""
    WITH hasil AS (
        SELECT 'event_id tidak null' AS aturan,
               COUNT(*) FILTER (WHERE event_id IS NULL) AS pelanggaran
        FROM dataset_bersih
        UNION ALL
        SELECT 'event_id unik',
               COUNT(*) - COUNT(DISTINCT event_id)
        FROM dataset_bersih
        UNION ALL
        SELECT 'session_id tidak null',
               COUNT(*) FILTER (WHERE session_id IS NULL)
        FROM dataset_bersih
        UNION ALL
        SELECT 'traffic_source hanya MOBILE atau WEB',
               COUNT(*) FILTER (WHERE traffic_source NOT IN ('MOBILE', 'WEB'))
        FROM dataset_bersih
        UNION ALL
        SELECT 'event_time dalam rentang 2015-2022',
               COUNT(*) FILTER (WHERE event_time < TIMESTAMPTZ '2015-01-01'
                                   OR event_time >= TIMESTAMPTZ '2023-01-01')
        FROM dataset_bersih
    )
    SELECT aturan, pelanggaran,
           CASE WHEN pelanggaran = 0 THEN 'LULUS' ELSE 'PERIKSA' END AS status
    FROM hasil
""").pl()
validasi

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

aturan,pelanggaran,status
str,i64,str
"""event_id tidak null""",0,"""LULUS"""
"""event_id unik""",0,"""LULUS"""
"""session_id tidak null""",0,"""LULUS"""
"""traffic_source hanya MOBILE at…",0,"""LULUS"""
"""event_time dalam rentang 2015-…",0,"""LULUS"""


In [23]:
jumlah_akhir = connection.execute('SELECT COUNT(*) FROM dataset_bersih').fetchone()[0]
print(f'Baris sebelum cleaning: {jumlah_baris:,}')
print(f'Baris sesudah cleaning: {jumlah_akhir:,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baris sebelum cleaning: 12,833,602
Baris sesudah cleaning: 12,833,602


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

## Catatan Milestone 2

* **Dimensi kualitas data yang paling menonjol:**
  * *Completeness:* Ditemukan 8.544.062 nilai null (66,58%) hanya pada kolom `event_metadata`, sedangkan 5 kolom kunci lainnya memiliki 0% null. Setelah dicek per `event_name`, null pada `event_metadata` terbukti bersifat **struktural** (100% null pada `CLICK`, `HOMEPAGE`, `SCROLL`, `ITEM_DETAIL`, `PROMO_PAGE`, dan 0% null pada `ADD_TO_CART`, `SEARCH`, `BOOKING`, `ADD_PROMO`).
  * *Uniqueness:* Tidak ditemukan duplikat pada `event_id`.
  * *Anomali/Outlier:* Pada agregasi 895.203 sesi, distribusi `jumlah_event` per sesi memiliki Q1 = 6, Q3 = 18, IQR = 12, batas atas IQR = 36, dan p99 = 65. Sebanyak 49.482 sesi (5,53%) berada di atas batas atas IQR (maksimum 650 event/sesi).
* **Strategi yang dipilih:**
  * *Missing values:* Nilai null pada `event_metadata` dipertahankan (tidak dilakukan `fill_null` maupun penghapusan baris) agar tidak merusak makna struktural data, sementara baris dengan null pada 5 kolom kunci disaring (0 baris dibuang).
  * *Outlier:* Sesi dengan jumlah event tinggi hanya dicatat/dilaporkan tanpa dihapus atau di-*clip*, karena distribusi data cacah memang condong ke kanan (*right-skewed*) dan belum ada bukti valid bahwa sesi tersebut merupakan bot.
  * *Tipe data:* Kolom `event_time` dipastikan bertipe `Datetime(us, UTC)`.
* **Perubahan jumlah baris & ukuran file:** Baris sebelum dan sesudah cleaning tetap **12.833.602 baris** (0 baris dihapus). Ukuran file berubah dari 1.715,3 MB menjadi 1.766,6 MB (`click_stream_bersih.csv`).
* **Aturan validasi & hasil:** Pengujian ulang menggunakan DuckDB SQL pada file output untuk 5 aturan (`event_id` tidak null, `event_id` unik, `session_id` tidak null, `traffic_source` hanya `MOBILE`/`WEB`, dan `event_time` dalam rentang 2015–2022) seluruhnya menghasilkan **0 pelanggaran (LULUS)**.